# edit 赛道 · 数据集/题目分析（for 题目构造）

合并自原 `eval_analysis.ipynb`（抽样 + 分布 + 展示）与 `eval_review.ipynb`（题库审阅）：
1. **评测集构建**：委托同目录 `eval_sample.py`（SQL 过滤 + (L1,L2) 分支 √配额分层抽样，每次全量重抽覆盖写 `data/samples.jsonl`；默认口径 = 质量门 + 编辑适配门（主体显著 + 短边下限），`--no-edit-gate` 可关）
2. **评测集分布分析 / 抽样展示**：只读 `data/samples.jsonl`
3. **题库审阅**：只读 `data/synth_edit/questions.jsonl`（`eval_synthesize.py` 产物；未出题时对应格自动跳过）

> 运行：菜单 Run All，或逐 Cell 运行。抽样与题库过滤参数集中在各 cell 顶部，改完重跑该格即可。

In [ ]:
import sys, subprocess
import json as _json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML
plt.rcParams['font.family'] = ['Noto Sans CJK SC', 'DejaVu Sans']  # CJK 优先，避免中文豆腐块
plt.rcParams['axes.unicode_minus'] = False

## 评测集构建（eval_sample 分层抽样）

抽样直接委托同目录 `eval_sample.py`（机制与全部参数见脚本头 docstring 与 `--help`）：**每次运行全量重抽**，覆盖写 `data/samples.jsonl` 并清除 `data/images/` 旧样本拷贝（样本分布跟随最新数据）。

In [ ]:
# ---- 评测抽样：委托同目录 eval_sample.py（每次全量重抽，覆盖 data/samples.jsonl）----
EVAL_N           = 1000      # ← 改这里：抽取张数（直接抽该数量，覆盖写清单）
EVAL_FILTER      = "quality >= 8 AND identity = true AND focus >= 7 AND least(width, height) >= 512"
                             # ← 改这里：候选过滤条件（duckdb SQL WHERE，作用于清单）
EVAL_PER_INST    = 2         # ← 改这里：每实例限张数
EVAL_SEED        = 20260823  # ← 改这里：抽样种子（固定种子 → 可复现）
EVAL_DRY_RUN     = False     # ← 改这里：True=只打印配额分配不落盘
EVAL_OUT         = None      # ← 改这里：样本清单输出路径；None=脚本默认（data/samples.jsonl）
EVAL_IMG_DIR     = None      # ← 改这里：图片拷贝目录；None=脚本默认（data/images）

SAMPLES  = Path(EVAL_OUT) if EVAL_OUT else Path('data/samples.jsonl')

cmd = [sys.executable, str(Path('eval_sample.py').resolve()),
       '--n', str(EVAL_N), '--filter', EVAL_FILTER,
       '--per-instance', str(EVAL_PER_INST), '--seed', str(EVAL_SEED)]
if EVAL_OUT:
    cmd += ['--out', str(EVAL_OUT)]
if EVAL_IMG_DIR:
    cmd += ['--img-dir', str(EVAL_IMG_DIR)]
if EVAL_DRY_RUN:
    cmd.append('--dry-run')
assert subprocess.run(cmd).returncode == 0, 'eval_sample.py 异常退出'   # 脚本自推仓库根，不依赖 cwd

rows = [_json.loads(l) for l in open(SAMPLES, encoding='utf-8') if l.strip()]
sdf = pd.DataFrame(rows)
EVAL_DIR = SAMPLES.parent                        # 图片按清单内相对 image 字段 + 清单所在目录解析
print(f'评测集: {len(sdf):,} 样本 / {sdf.instance.nunique():,} 实例 → {SAMPLES.resolve()}')
with pd.option_context('display.max_colwidth', 60):
    display(sdf[['sample_id', 'instance', 'l1', 'l2', 'quality', 'focus', 'kb_match']].head(10))

In [ ]:
# ---- 评测集分布分析（只读 samples.jsonl；复用上一格 sdf）----
per_inst = sdf.instance.value_counts()
print(f'总图数: {len(sdf):,}   去重实例: {sdf.instance.nunique():,}   覆盖 (L1,L2) 分支: {sdf.groupby(["l1","l2"]).ngroups}')
print('每实例图数分布: ' + '   '.join(f'{k} 张: {v:,} 实例' for k, v in per_inst.value_counts().sort_index().items()))

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
# ① L1 分支分布
l1 = sdf.l1.value_counts()
axes[0][0].barh(l1.index[::-1], l1.values[::-1], color='#8ab')
for i, v in enumerate(l1.values[::-1]):
    axes[0][0].text(v, i, f' {v:,}', va='center', fontsize=8)
axes[0][0].set_title(f'L1 分支分布（{len(l1)} 个）')
# ② (L1,L2) 分支 Top20
br = sdf.groupby(['l1', 'l2']).size().sort_values().tail(20)
axes[0][1].barh([' / '.join(ix) for ix in br.index], br.values, color='#7a9')
axes[0][1].tick_params(axis='y', labelsize=7)
axes[0][1].set_title(f'(L1,L2) 分支样本量 Top20（共 {sdf.groupby(["l1","l2"]).ngroups} 分支）')
# ③~⑤ 打分字段分布
for ax, f in [(axes[0][2], 'quality'), (axes[1][0], 'focus'), (axes[1][1], 'kb_match')]:
    vals = pd.to_numeric(sdf[f], errors='coerce').dropna()
    lo, hi = float(vals.min()), float(vals.max())
    ax.hist(vals, bins=(np.linspace(lo, hi, 21) if hi > lo else 1), edgecolor='white', alpha=0.85)
    ax.axvline(vals.mean(), color='red', ls='--', lw=1, label=f'mean={vals.mean():.2f}')
    ax.legend(fontsize=8)
    ax.set_title(f'{f}  min={lo:g}  max={hi:g}  null={len(sdf)-len(vals)}')
# ⑥ 短边分布
se = np.minimum(sdf.width, sdf.height).dropna()
axes[1][2].hist(se, bins=40, edgecolor='white', color='#97a', alpha=0.85)
axes[1][2].legend(fontsize=8)
axes[1][2].set_title(f'图片短边分布（中位 {se.median():.0f}px）')
plt.tight_layout(); plt.show()

In [ ]:
# ---- 评测集抽样展示（图卡；复用 sdf / EVAL_DIR）----
N_CASES = 8     # ← 改这里：抽几张展示

show = sdf.sample(min(N_CASES, len(sdf)), random_state=42)
for _, r in show.iterrows():
    ip = EVAL_DIR / r['image']
    img = (f'<img src="{ip}" loading="lazy" style="max-height:280px;max-width:380px;object-fit:contain;background:#f6f6f6">'
           if ip.is_file() else '<div style="color:#c00">图缺失</div>')
    kv = ''.join(f'<div style="font-size:12px;margin:1px 0"><b>{k}</b>: {v}</div>' for k, v in [
        ('sample', f'{r["sample_id"]} · {r["instance"]}'),
        ('branch', f'{r["l1"]} / {r["l2"]}'),
        ('metrics', f'quality={r["quality"]}  focus={r["focus"]}  kb_match={r["kb_match"]}  {r["width"]}×{r["height"]}'),
        ('caption', str(r.get('caption', ''))[:220]),
    ])
    display(HTML(f'<div style="display:flex;gap:12px;border:1px solid #ddd;padding:8px;margin:8px 0;align-items:flex-start">'
                 f'{img}<div style="min-width:0">{kv}</div></div>'))

## 题库审阅（for 题目构造）

只读 `eval_synthesize.py` 出题产物；审阅要点：① 答案/改动是否真的由图外知识决定（对照每题的证据审计 `visible_facts`）；② `expected_failure_modes` 是否具体、可判；③ 事实性（合成模型自报 `needs_verification` 全为 0，需人工核）。

In [ ]:
# ---- 题库加载（eval_synthesize.py 出题产物；不存在则本段跳过）----
QFILE = Path('data/synth_edit/questions.jsonl')
if not QFILE.exists():
    qs = []
    print(f'!! 题库不存在：{QFILE} —— 先跑 eval_synthesize.py 出题；只看样本分析可跳过后续格')
else:
    qs = [_json.loads(l) for l in QFILE.open(encoding='utf-8')]
    by_sample = {}
    for q in qs:
        by_sample.setdefault(q['_sample_image'], []).append(q)
    print(f'{len(qs)} 题 / {len(by_sample)} 个样本')

In [ ]:
# ---- 题库打印工具（只读展示）----
from IPython.display import Image as _NBImage

W = 74

def _pr(items, key):
    for p in items or []:
        if isinstance(p, dict):
            w = p.get('weight')
            txt = p.get(key) or p.get('point') or p.get('check') or ''
            print(f"  - {('[' + str(w) + '] ') if w is not None else ''}{txt}")
        else:
            print(f"  - {p}")

def _print_question(q):
    task = (q.get('task') or '?').upper()
    print('\n' + '-' * W)
    print(f"[{task} | {q.get('difficulty', '?')} | {q.get('qid', '?')} | "
          f"知识维度: {q.get('knowledge_dim', '?')}]"
          + (f" | edit_type: {q.get('edit_type')} | suite: {q.get('suite')}" if q.get('edit_type') else ''))
    print(f"探针维度: {'、'.join(q.get('probe_dims') or [])}")
    prompt = q.get('stem') or q.get('edit_instruction') or q.get('gen_prompt') or ''
    print(f"\n[指令]\n{prompt}")
    if q.get('choices'):
        print(f"\n[选项]\n{q['choices']}")
    if q.get('reference_answer'):
        print(f"\n[参考答案]\n{q['reference_answer']}")
    if q.get('rubric'):
        print('\n[判分要点]'); _pr(q.get('rubric'), 'point')
    if q.get('expected_changes'):
        print('\n[执行到位要点]'); _pr(q.get('expected_changes'), 'point')
    if q.get('preserved_elements'):
        print('\n[保持一致要点]'); _pr(q.get('preserved_elements'), 'point')
    if q.get('implicit_checks'):
        print('\n[隐含知识校验点]'); _pr(q.get('implicit_checks'), 'check')
    print(f"\n[推理链] {q.get('reasoning_chain', '')}")
    fm = q.get('expected_failure_modes') or []
    if fm:
        print('\n[预期失败模式]')
        for m in fm:
            print(f"  * {m}")
    ea = q.get('evidence_audit') or {}
    if ea:
        print('\n[证据审计]')
        print('  visible_facts:', '；'.join(ea.get('visible_facts') or []))
        print('  answerable_by_image:', '；'.join(ea.get('answerable_by_image') or []))
    if q.get('needs_verification'):
        print('  !!! 需人工核验')

def _print_sample(img_rel, group, img_width=480):
    label = group[0].get('_query_label', '')
    sid = img_rel.split('/')[1][:4]
    img_path = EVAL_DIR / img_rel

    print('\n' + '=' * W)
    print(f'>>> 样本 {sid} | {label} | {len(group)} 题')
    print('=' * W)
    try:
        meta = next(_json.loads(l) for l in (EVAL_DIR / 'samples.jsonl').open(encoding='utf-8')
                    if _json.loads(l)['image'] == img_rel)
        print('caption:', meta.get('caption', ''))
    except (StopIteration, OSError):
        pass
    if img_path.exists():
        display(_NBImage(filename=str(img_path), width=img_width))
    else:
        print('!! 缺图:', img_path)
    for q in group:
        _print_question(q)

In [ ]:
# 可用取值速览：过滤参数能填什么，看这里
from collections import Counter

def _cnt(items):
    return ' | '.join(f'{k}:{v}' for k, v in sorted(Counter(items).items()))

if not qs:
    print('题库为空，跳过（先跑 eval_synthesize.py 出题）')
else:
    print('difficulty   :', _cnt(q.get('difficulty') for q in qs))
    print('knowledge_dim:', _cnt(q.get('knowledge_dim') for q in qs))
    print('probe_dims   :', _cnt(p for q in qs for p in (q.get('probe_dims') or [])))
    print('sample 标签  :', _cnt(sorted({q['_query_label'] for q in qs})))
    print('qid          :', ', '.join(q['qid'] for q in qs))

In [ ]:
# ====== 过滤 / 抽样参数（改完重跑本 cell 即可；空值 = 不过滤）======
QIDS = []              # 精确指定题目
SAMPLES = []           # 按样本筛：样本号前缀或 query 标签，如 ['0001', '厦门方特梦幻王国']
DIFFICULTIES = []      # 难度：'L1' / 'L2' / 'L3'
KNOWLEDGE_DIMS = []    # 知识维度，如 ['physics_commonsense', 'film_anime_game']
PROBE_DIMS = []        # 探针维度（题目命中任一即保留），如 ['hallucination_resist']
ONLY_NEEDS_VERIFY = False   # True = 只看自报需人工核验的题
SAMPLE_N = 0           # >0 时按样本抽：随机抽 N 个样本（与上面过滤叠加），0 = 不抽
SEED = 7               # 抽样随机种子，固定可复现；换种子换一批
IMG_WIDTH = 480        # 图片显示宽度
# ===================================================================

def _keep(q):
    if QIDS and q['qid'] not in QIDS:
        return False
    if SAMPLES:
        sid = q['_sample_image'].split('/')[1][:4]
        if not (sid in SAMPLES or q.get('_query_label') in SAMPLES):
            return False
    if DIFFICULTIES and q.get('difficulty') not in DIFFICULTIES:
        return False
    if KNOWLEDGE_DIMS and q.get('knowledge_dim') not in KNOWLEDGE_DIMS:
        return False
    if PROBE_DIMS and not (set(PROBE_DIMS) & set(q.get('probe_dims') or [])):
        return False
    if ONLY_NEEDS_VERIFY and not q.get('needs_verification'):
        return False
    return True

if not qs:
    print('题库为空，跳过（先跑 eval_synthesize.py 出题）')
else:
    _sel = [q for q in qs if _keep(q)]
    if SAMPLE_N > 0:
        import random
        pool = sorted({q['_sample_image'] for q in _sel})
        if len(pool) > SAMPLE_N:
            pool = set(random.Random(SEED).sample(pool, SAMPLE_N))
        _sel = [q for q in _sel if q['_sample_image'] in pool]

    print(f'命中 {len(_sel)} 题 / {len({q["_sample_image"] for q in _sel})} 个样本（全库 {len(qs)} 题）')
    for _img, _group in sorted({k: [q for q in _sel if q['_sample_image'] == k]
                                for k in {q['_sample_image'] for q in _sel}}.items()):
        _print_sample(_img, _group, IMG_WIDTH)